# Transformer的文本分类应用

# 要求
## 1. 先将代码补充完整，运行得到结果
## 2. Gated Attention for Large Language Models: Non-linearity, Sparsity, and Attention-Sink-Free 获得NeurIPS 2025 Best Paper Awards，用一个极其简单的改动，同时解决了Transformer架构的多个痛点。
## 在attention的value投影和输出投影之间，插一个sigmoid门控。
### 在连续线性变换中引入非线性
### 让模型获得了"选择性沉默"的能力
### 消除了困扰LLM多年的Attention Sink现象
## 尝试将其方式应用于此应用


### 导入库

In [ ]:
!pip install datasets transformers

In [ ]:
import math
import time
from datetime import datetime
import torch
# from functorch.einops import rearrange
from einops import rearrange
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from datasets import load_dataset
from transformers import BertTokenizerFast

定义设备 tensorboard 记录器

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter(log_dir='./log/' + time.strftime('%m-%d_%H.%M', time.localtime()))

### 定义超参数

In [ ]:
batch_size = 16
epochs_num = 5
lr = 1e-5

分词器和词表,注意这里的词表,仅仅使用了一个分词功能和id映射功能,没有使用到词嵌入的映射

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

### 位置编码器：此次有填空

In [ ]:
class PositionalEncoding(nn.Module):
    """
    位置编码器
    """

    def __init__(self, d_model, max_len=512, device=None):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        self.register_buffer('positional_encoding', self.encoding)

        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        # "位置编码"
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)

        self.encoding = self.encoding.unsqueeze(0)  # Add batch dimension
        self.encoding = self.encoding.to(device)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1)]

### 多头注意力机制 ：此次有填空

In [ ]:
class AttentionMulti(nn.Module):
    def __init__(self, embedding_feature_dim, attention_feature_dim, heads, dropout=0.):
        """
        :param embedding_feature_dim: 词嵌入特征维度: 例300
        :param attention_feature_dim: 注意力特征维度(每一个头),即dk 例:128
        :param heads: 多头注意力的头数 例:8
        """
        super(AttentionMulti, self).__init__()
        # qkv合一,形状一致,深度为attention_feature_dim * heads * 3
        self.w_qkv = nn.Linear(embedding_feature_dim, attention_feature_dim * heads * 3)
        self.heads = heads
        self.dk = torch.tensor(attention_feature_dim / heads, dtype=torch.float32)
        self.attention_to_embedding = nn.Linear(attention_feature_dim * heads, embedding_feature_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(attention_feature_dim * heads, attention_feature_dim * heads)

    def forward(self, x, mask):
        # 6x512x300 -> 6x512x(128x8x3)
        qkv = self.w_qkv(x)
        # 沿着qkv切开
        qkv = qkv.chunk(3, dim=-1)
        # 分离出heads提到前面来
        # b (batch_size); h (heads)头数 ;n (sequence length)序列长度(上下文长度);d (dimension) 注意力特征维度
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)

        # 6x8x512x128 叉乘 6x8x512x128 ->6x8x512x512,得128维组合的相关性
        # "注意力机制计算公式"，提示：注意k的转置，k*v利用torch.matmul函数
        attention_tensor = torch.matmul(q,k.transpose(-2,-1))
        #  /sqrt(dk)
        attention_tensor = attention_tensor / torch.sqrt(self.dk)

        # 在计算注意力权重时，将填充部分的权重设置为负无穷大，以使 softmax 计算后变为零。从而使填充不会影响到其他数据
        if mask is not None:
            # 使用填充掩码调整注意力权重
            mask = mask.unsqueeze(1).unsqueeze(2)  # Shape: [b, 1, 1, n]
            attention_tensor = attention_tensor.masked_fill(mask == 0, float('-inf'))

        # 对结果做softmax得到概率值(6x8x512x512) 注意维度的选择
        attention_tensor = torch.softmax(attention_tensor,dim=-1)
        attention_tensor = self.dropout(attention_tensor)

        # 做叉乘,得6x8x512x128,即在这个注意力网络中,对每一个的词向量的改变值.
        #  v * attention_tensor
        out = torch.matmul(attention_tensor,v)
        # 将多头获取到的变化合并到最后的改变特征值维度,6x8x512x128 -> 6x512x(128x8)
        # 即每个头都为原始的词嵌入提供了128维度的变化量,总计8个头,故而提供128x8,所以合并这些变化量
        out = rearrange(out, 'b h n d -> b n (h d)', h=self.heads)
        gate = torch.sigmoid(self.gate(out))
        out = out * gate
        out = self.attention_to_embedding(out)

        out = self.dropout(out)
        # 输出尺寸和输入一致 6x512x300
        return out

### transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(self, depth, embedding_feature_dim, attention_feature_dim, heads, mlp_feature_dim, dropout=0.):
        """
        :param depth: 模型深度:例如3
        :param embedding_feature_dim: 词嵌入特征维度: 例300
        :param attention_feature_dim: 注意力特征维度(每一个头),即dk 例:128
        :param heads: 多头注意力的头数 例:8
        :param mlp_feature_dim: mlp的特征维度:例1200
        """
        super(Transformer, self).__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            # 多头注意力
            attention = AttentionMulti(embedding_feature_dim, attention_feature_dim, heads)
            # 每个MLP一个ln,2个全连接,一个激活函数构成
            mlp = nn.Sequential(
                nn.LayerNorm(embedding_feature_dim),
                nn.Linear(embedding_feature_dim, mlp_feature_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(mlp_feature_dim, embedding_feature_dim),
                nn.Dropout(dropout),
            )
            self.layers.append(nn.ModuleList([attention, mlp]))
        # 深度次数后的的Transformer,加上一个层归一化
        self.norm = nn.LayerNorm(embedding_feature_dim)

    def forward(self, x, mask):
        for attention, mlp in self.layers:
            x = x + attention(x, mask)
            x = x + mlp(x)
        return self.norm(x)

### 基于transformer的文本分类模型

In [ ]:
class TextClassifyTransformer(nn.Module):
    def __init__(self, class_num, vocab_size, sequence_length, depth, embedding_feature_dim, attention_feature_dim,
                 heads, mlp_feature_dim, dropout=0., emb_dropout=0., device=None):
        """
        :param class_num: 分类数量
        :param vocab_size: 词表大小,
        :param sequence_length: 序列长度(上下文长度)
        :param depth: Transformer模型深度:例如3,即自注意力模块和mlp重复多少次
        :param embedding_feature_dim: 词嵌入特征维度: 例300
        :param attention_feature_dim: 注意力特征维度(每一个头),即dk 例:128
        :param heads: 多头注意力的头数 例:8
        :param mlp_feature_dim: mlp的特征维度:例1200
        :param dropout: 丢弃率.这里一共会使用4处,1.注意力矩阵计算完成,对注意力矩阵执行. 2.从注意力转为词向量的全连接层之后 3.mlp网络第一次全连接,激活后,4.mlp第二次全连接后
        :param emb_dropout: 对抗过拟合,此处为编码完成之后的dropout,即对词嵌入后执行的
        """
        super(TextClassifyTransformer, self).__init__()
        # 建立词嵌入层,词表长度,嵌入维度
        self.embedding = nn.Embedding(vocab_size, embedding_feature_dim)
        # 归一化
        self.ln1 = nn.LayerNorm(embedding_feature_dim)
        # 位置编码器
        self.pos_encoder = PositionalEncoding(d_model=embedding_feature_dim, max_len=sequence_length, device=device)
        self.emb_dropout = nn.Dropout(emb_dropout)

        # Transformer模块
        self.transformer = Transformer(depth, embedding_feature_dim, attention_feature_dim, heads, mlp_feature_dim,
                                       dropout)

        self.fc = nn.Linear(embedding_feature_dim, class_num)

    def forward(self, x, mask):
        # 执行词嵌入,将输入input转对应词向量
        x = self.embedding(x)

        # 嵌入位置
        x = self.pos_encoder(x)

        x = self.emb_dropout(x)

        # 执行layerNorm,即 6x512x300 的300维度执行归一化,即对一句话的每一个词向量执行归一化,2组权重参数β和γ
        x = self.ln1(x)

        # 执行Transformer模块,得 6x512x300
        x = self.transformer(x, mask)

        # 执行分类操作,直接执行 300->4的分类是缺少意义的,是针对一句话的每一个词(512)分别求分类,需要在第1维度作平均,平均一句话的内容
        x = x.mean(dim=1)

        # 全连接执行分类操作,得分类的logtis值
        return self.fc(x)

### 训练

#### 构造模型

In [ ]:
model = TextClassifyTransformer(4, tokenizer.vocab_size,
                                   sequence_length=512,
                                   depth=6, embedding_feature_dim=300,
                                   attention_feature_dim=128, heads=4,
                                    mlp_feature_dim=1200, dropout=0.1,
                                    emb_dropout=0.1, device=device)
model.to(device)

TextClassifyTransformer(
  (embedding): Embedding(30522, 300)
  (ln1): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
  (pos_encoder): PositionalEncoding()
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): AttentionMulti(
          (w_qkv): Linear(in_features=300, out_features=1536, bias=True)
          (attention_to_embedding): Linear(in_features=512, out_features=300, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (gate): Linear(in_features=512, out_features=512, bias=True)
        )
        (1): Sequential(
          (0): LayerNorm((300,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=300, out_features=1200, bias=True)
          (2): GELU(approximate='none')
          (3): Dropout(p=0.1, inplace=False)
          (4): Linear(in_features=1200, out_features=300, bias=True)
          (5): Dropout(p=0.1, inplace=False)
        )
      )
  

#### 损失函数,优化器

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.7)

#### 加载数据， 注意resources文件夹

In [ ]:
raw_datasets = load_dataset("ag_news")

def collate_fn(batch):
    labels = torch.tensor([item['label'] + 1 for item in batch])
    texts = [item['text'] for item in batch]
    return labels, texts

train_data_loader = DataLoader(
    dataset=raw_datasets['train'],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

valid_data_loader = DataLoader(
    dataset=raw_datasets['test'],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

train_data_size = len(raw_datasets['train'])
valid_data_size = len(raw_datasets['test'])

print(f"数据集加载成功！训练集: {train_data_size}, 测试集: {valid_data_size}")

数据集加载成功！训练集: 120000, 测试集: 7600


#### 训练

In [ ]:
# minibatch计数
def do_train(epoch):
    model.train()
    epoch_loss_sum = 0
    epoch_acc_sum = 0
    train_batch_index = 0
    ix = 0
    for label, texts in train_data_loader:
        label = label - 1
        label = label.to(device)

    # 分词,找词表对应下标,还有填充,这里一个方法一起做了
        encoded_batch = tokenizer(
                texts,
                max_length=512,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
            # token的id,6x512
        input_ids = encoded_batch['input_ids'].to(device)
        attention_mask = encoded_batch['attention_mask'].to(device)

        with (torch.set_grad_enabled(True)):
            # 传入填充过的mask,计算分类
            output = model(input_ids, attention_mask)

            loss = criterion(output, label)
            _, prediction = torch.max(output, 1)
                # 执行反向传播
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # 计算当前小批量的正确率
            batch_acc = prediction.eq(label).sum()
            # 计算批次准确率
            # 当前批量大小,由于数据集可能都不足一个batch_size,所以要获取到准确的batch_size
            now_batch_size = label.shape[0]

            # 将损失乘上size是为了在计算平均损失时，考虑到每个样本对损失的贡献。
            epoch_loss_sum += (loss * now_batch_size)
            epoch_acc_sum += batch_acc
            # 记录事件
            print('[%d/%d][%d/%d]\t%s\t loss: %.4f\t acc_rate: %.4f'
                      % (epoch, epochs_num, ix, train_data_size, datetime.now(), loss.item(), batch_acc / now_batch_size))

            train_batch_index += 1
            ix += batch_size

            epoch_loss = epoch_loss_sum / train_data_size
            epoch_acc_rate = epoch_acc_sum / train_data_size
    return epoch_loss, epoch_acc_rate

#### 模型验证

In [ ]:
def do_valid(epoch):
    """
        验证过程
        :param epoch:
        :return:
    """
    model.eval()
    # 进入推理模式,不计算梯度
    epoch_loss_sum = 0
    epoch_acc_sum = 0
    valid_batch_index = 0

    ix = 0
    for label, texts in valid_data_loader:
        label = label - 1
        label = label.to(device)

        # 分词,找词表对应下标,还有填充,这里一个方法一起做了
        encoded_batch = tokenizer(
                texts,
                max_length=512,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
            # token的id,6x512
        input_ids = encoded_batch['input_ids'].to(device)
        attention_mask = encoded_batch['attention_mask'].to(device)

        with (torch.set_grad_enabled(False)):
            # 传入填充过的mask,计算分类
            output = model(input_ids, attention_mask)

            loss = criterion(output, label)
            _, prediction = torch.max(output, 1)
            # 计算当前小批量的正确率
            batch_acc = prediction.eq(label).sum()
            # 计算批次准确率
            # 当前批量大小,由于数据集可能都不足一个batch_size,所以要获取到准确的batch_size
            now_batch_size = label.shape[0]

            # 将损失乘上size是为了在计算平均损失时，考虑到每个样本对损失的贡献。
            epoch_loss_sum += (loss * now_batch_size)
            epoch_acc_sum += batch_acc


            print('[%d/%d][%d/%d]\t%s\t loss: %.4f'
                % (epoch, epochs_num, ix,
                   valid_data_size, datetime.now(), loss.item()))


            ix += batch_size

    epoch_loss = epoch_loss_sum / valid_data_size
    epoch_acc_rate = epoch_acc_sum / valid_data_size
    return epoch_loss, epoch_acc_rate

#### 模型训练

In [ ]:
best_epoch_acc_rate_valid = 0

# 重置批次计数
train_batch_index = 0
valid_batch_index = 0
# 更新最优参数
for epoch in range(epochs_num):
    epoch_loss_train, epoch_acc_rate_train = do_train(epoch)
    # 更新优化器
    lr_scheduler.step()
    # 执行验证
    epoch_loss_valid, epoch_acc_rate_valid = do_valid(epoch)
    print(f"epoch {epoch}/{epochs_num - 1} : "
          f"epoch_loss_train:{epoch_loss_train:.4f};epoch_acc_rate_train:{epoch_acc_rate_train:.4f}; "
          f"epoch_loss_valid:{epoch_loss_valid:.4f};epoch_acc_rate_valid:{epoch_acc_rate_valid:.4f}; ")
    if epoch_acc_rate_valid >= best_epoch_acc_rate_valid:
        best_epoch_acc_rate_valid = epoch_acc_rate_valid
    print("best_epoch_acc_rate_valid", best_epoch_acc_rate_valid)


流式输出内容被截断，只能显示最后 5000 行内容。
[4/5][47648/120000]	2026-05-25 04:34:51.878251	 loss: 0.3082	 acc_rate: 0.9375
[4/5][47664/120000]	2026-05-25 04:34:52.153981	 loss: 0.1619	 acc_rate: 0.9375
[4/5][47680/120000]	2026-05-25 04:34:52.427558	 loss: 0.2061	 acc_rate: 0.8750
[4/5][47696/120000]	2026-05-25 04:34:52.699036	 loss: 0.2229	 acc_rate: 0.9375
[4/5][47712/120000]	2026-05-25 04:34:52.974459	 loss: 0.1312	 acc_rate: 0.9375
[4/5][47728/120000]	2026-05-25 04:34:53.247416	 loss: 0.2239	 acc_rate: 0.9375
[4/5][47744/120000]	2026-05-25 04:34:53.525086	 loss: 0.3919	 acc_rate: 0.8750
[4/5][47760/120000]	2026-05-25 04:34:53.798346	 loss: 0.3868	 acc_rate: 0.8750
[4/5][47776/120000]	2026-05-25 04:34:54.075310	 loss: 0.7257	 acc_rate: 0.8750
[4/5][47792/120000]	2026-05-25 04:34:54.352079	 loss: 0.8884	 acc_rate: 0.7500
[4/5][47808/120000]	2026-05-25 04:34:54.640639	 loss: 0.1847	 acc_rate: 0.9375
[4/5][47824/120000]	2026-05-25 04:34:54.902504	 loss: 0.4839	 acc_rate: 0.6875
[4/5][47840/120000]	2026-